# regression lineaire multiple - estimation de prix immobilier

**user story** : en tant qu'agent immobilier, je veux obtenir une estimation automatique du prix d'un bien en fonction de sa surface, son type et son quartier, afin de conseiller un vendeur sur le prix de mise en vente realiste.

**modele** : regression multiple (surface + type de bien + code postal)

**variables** :
- surface en m²
- type : appartement ou maison
- code postal : 83000 (centre), 83100 (ouest), 83200 (est)

**scoring** :
- **sous-evalue** : prix demande < -15% du prix estime (opportunite)
- **prix marche** : prix demande entre -15% et +15% du prix estime (correct)
- **surevalue** : prix demande > +15% du prix estime (negociation necessaire)

In [1]:
# imports
import csv
import sys
from pathlib import Path

# ajouter le dossier parent au path pour importer les modules
sys.path.insert(0, str(Path.cwd().parent))

from analysis.regression_multiple import (
    least_squares_fit_multiple,
    r_squared_multiple,
    predict_multiple
)

## chargement des donnees et entrainement du modele

In [ ]:
# chargement des donnees dvf nettoyees
surfaces = []
prix = []
types = []  # 0 = appartement, 1 = maison
codes_postaux = []

csv_path = Path.cwd().parent / "data" / "DVF-83-Toulon-2024-2025advanced.csv"

with open(csv_path, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter=";")
    for row in reader:
        try:
            s = float(row["surface_totale"])
            p = float(row["valeur_fonciere"])
            t = 1.0 if row["type_local"] == "Maison" else 0.0
            cp = row["code_postal"]
            
            surfaces.append(s)
            prix.append(p)
            types.append(t)
            codes_postaux.append(cp)
        except (ValueError, TypeError, KeyError):
            continue

print(f"donnees chargees : {len(surfaces)} transactions")
print(f"appartements : {sum(1 for t in types if t == 0)}")
print(f"maisons : {sum(1 for t in types if t == 1)}")
print()

# repartition par code postal
from collections import Counter
cp_counter = Counter(codes_postaux)
print("repartition par quartier :")
for cp, count in sorted(cp_counter.items()):
    print(f"  {cp} : {count} transactions")

In [ ]:
# preparation des features pour regression multiple avec code postal
# encodage one-hot : 83100 est la reference
# x = [surface, is_maison, is_83000, is_83200]
x_multi = []
for s, t, cp in zip(surfaces, types, codes_postaux):
    is_83000 = 1.0 if cp == '83000' else 0.0
    is_83200 = 1.0 if cp == '83200' else 0.0
    x_multi.append([s, t, is_83000, is_83200])

# entrainement du modele (optimise pour eviter timeout)
print("entrainement du modele en cours...")
beta = least_squares_fit_multiple(x_multi, prix, learning_rate=0.05, num_iterations=3000)
r2 = r_squared_multiple(beta, x_multi, prix)

print("\nmodele entraine !")
print(f"\nformule :")
print(f"prix = {beta[0]:,.0f}")
print(f"     + {beta[1]:,.2f} * surface")
print(f"     + {beta[2]:,.0f} * is_maison")
print(f"     + {beta[3]:,.0f} * is_cp83000")
print(f"     + {beta[4]:,.0f} * is_cp83200")
print()
print(f"r² = {r2:.4f} ({r2*100:.2f}% de la variance expliquee)")
print()
print("interpretation :")
print(f"- prix de base : {beta[0]:,.0f} euros")
print(f"- prix par m² : {beta[1]:,.2f} euros/m²")
print(f"- prime maison vs appartement : +{beta[2]:,.0f} euros")
print(f"- effet quartier 83000 (centre) : {beta[3]:+,.0f} euros vs 83100")
print(f"- effet quartier 83200 (est) : {beta[4]:+,.0f} euros vs 83100")

## fonction d'evaluation d'un bien

In [ ]:
def evaluer_bien(surface, type_bien, code_postal, prix_demande, beta, seuil=0.15):
    """
    evalue si un bien est sous-evalue, correctement evalue ou surevalue.
    
    parametres :
    - surface : surface en m²
    - type_bien : 'Appartement' ou 'Maison'
    - code_postal : '83000', '83100' ou '83200'
    - prix_demande : prix affiche dans l'annonce
    - beta : coefficients du modele
    - seuil : seuil d'acceptation (0.15 = ±15%)
    
    retourne : dictionnaire avec estimation et evaluation
    """
    # conversion des variables categoriques
    type_num = 1.0 if type_bien.lower() == 'maison' else 0.0
    is_83000 = 1.0 if code_postal == '83000' else 0.0
    is_83200 = 1.0 if code_postal == '83200' else 0.0
    
    # prediction du prix estime par le modele
    prix_estime = predict_multiple(beta, [surface, type_num, is_83000, is_83200])
    
    # calcul de l'ecart en pourcentage
    ecart_euros = prix_demande - prix_estime
    ecart_pct = (ecart_euros / prix_estime) * 100
    
    # scoring base sur le seuil
    if ecart_pct < -seuil * 100:
        evaluation = "SOUS-EVALUE"
        conseil = "opportunite ! prix inferieur au marche"
    elif ecart_pct > seuil * 100:
        evaluation = "SUREVALUE"
        conseil = "prix superieur au marche, negociation recommandee"
    else:
        evaluation = "PRIX MARCHE"
        conseil = "prix coherent avec le marche"
    
    return {
        'surface': surface,
        'type': type_bien,
        'code_postal': code_postal,
        'prix_demande': prix_demande,
        'prix_estime': prix_estime,
        'ecart_euros': ecart_euros,
        'ecart_pct': ecart_pct,
        'evaluation': evaluation,
        'conseil': conseil
    }


def afficher_evaluation(resultat):
    """affiche l'evaluation de maniere lisible."""
    print("="*60)
    print(f"bien : {resultat['type']} de {resultat['surface']:.0f} m²")
    print(f"quartier : {resultat['code_postal']}")
    print(f"prix demande : {resultat['prix_demande']:,.0f} euros")
    print("-"*60)
    print(f"prix estime (modele) : {resultat['prix_estime']:,.0f} euros")
    print(f"ecart : {resultat['ecart_euros']:+,.0f} euros ({resultat['ecart_pct']:+.1f}%)")
    print("-"*60)
    print(f"evaluation : {resultat['evaluation']}")
    print(f"conseil : {resultat['conseil']}")
    print("="*60)

# test rapide de la fonction
test = evaluer_bien(70, 'Appartement', '83000', 250000, beta)
afficher_evaluation(test)

## tests sur des exemples reels du dataset

on va prendre quelques transactions reelles et voir ce que le modele aurait predit

In [ ]:
# selection de quelques exemples reels varies
import random

# on prend 5 exemples aleatoires
random.seed(42)  # pour reproductibilite
indices = random.sample(range(len(surfaces)), 5)

print("exemples de biens reels du marche toulonnais :\n")

for i, idx in enumerate(indices, 1):
    surface_real = surfaces[idx]
    prix_real = prix[idx]
    type_real = 'Maison' if types[idx] == 1.0 else 'Appartement'
    cp_real = codes_postaux[idx]
    
    # on evalue le bien comme si c'etait une annonce
    resultat = evaluer_bien(surface_real, type_real, cp_real, prix_real, beta)
    
    print(f"\nexemple {i} :")
    afficher_evaluation(resultat)

## exemples specifiques : appartements vs maisons

In [ ]:
# comparaison des quartiers pour un meme bien
surface_test = 70
type_test = 'Appartement'
prix_test = 200000

print(f"comparaison pour un {type_test} de {surface_test} m² au prix de {prix_test:,} euros :\n")

for cp in ['83000', '83100', '83200']:
    eval_cp = evaluer_bien(surface_test, type_test, cp, prix_test, beta)
    print(f"si c'est dans le quartier {cp} :")
    afficher_evaluation(eval_cp)
    print()

## evaluation personnalisee : entrez vos propres valeurs

modifiez les valeurs ci-dessous pour tester votre propre bien

In [ ]:
# ===== MODIFIEZ CES VALEURS =====
ma_surface = 85  # surface en m²
mon_type = 'Appartement'  # 'Appartement' ou 'Maison'
mon_code_postal = '83000'  # '83000', '83100' ou '83200'
mon_prix = 280000  # prix demande en euros
# ================================

mon_evaluation = evaluer_bien(ma_surface, mon_type, mon_code_postal, mon_prix, beta)
print("\nvotre bien :")
afficher_evaluation(mon_evaluation)

## grille de predictions pour niddouillet

estimation de prix pour differentes surfaces (budget max 450k euros)

In [ ]:
# grille de prix pour conseillers niddouillet
surfaces_test = [30, 40, 50, 60, 70, 80, 90, 100]
code_postal_ref = '83000'  # quartier de reference pour la grille

print(f"grille d'estimations pour le marche toulonnais (quartier {code_postal_ref}) :")
print("\n" + "="*80)
print(f"{'Surface (m²)':<15} {'Appartement':<20} {'Maison':<20} {'Ecart':<15}")
print("="*80)

# encodage du code postal de reference
is_83000 = 1.0 if code_postal_ref == '83000' else 0.0
is_83200 = 1.0 if code_postal_ref == '83200' else 0.0

for surf in surfaces_test:
    prix_appart = predict_multiple(beta, [surf, 0.0, is_83000, is_83200])
    prix_maison = predict_multiple(beta, [surf, 1.0, is_83000, is_83200])
    ecart = prix_maison - prix_appart
    
    # indicateur budget niddouillet (max 450k)
    flag_appart = "" if prix_appart <= 450000 else " ⚠️"
    flag_maison = "" if prix_maison <= 450000 else " ⚠️"
    
    print(f"{surf:<15} {prix_appart:>10,.0f} €{flag_appart:<8} {prix_maison:>10,.0f} €{flag_maison:<8} +{ecart:>8,.0f} €")

print("="*80)
print("⚠️ = hors budget niddouillet (> 450k euros)")
print()

# comparaison entre quartiers
print("\ncomparaison des quartiers pour un appartement de 70m² :")
print("-"*60)
for cp in ['83000', '83100', '83200']:
    is_83000_cp = 1.0 if cp == '83000' else 0.0
    is_83200_cp = 1.0 if cp == '83200' else 0.0
    prix_cp = predict_multiple(beta, [70, 0.0, is_83000_cp, is_83200_cp])
    print(f"{cp} : {prix_cp:>10,.0f} euros")

## statistiques du modele

performance et fiabilite

In [9]:
# analyse des erreurs de prediction
# pas besoin de re-importer, deja fait en haut
from analysis.regression_multiple import error_multiple
from analysis.stats import mean, standard_deviation

# calcul des erreurs pour chaque transaction
erreurs = [error_multiple(beta, x_i, y_i) for x_i, y_i in zip(x_multi, prix)]
erreurs_absolues = [abs(e) for e in erreurs]
erreurs_pct = [(e / y_i) * 100 for e, y_i in zip(erreurs, prix)]

print("performance du modele sur les donnees d'entrainement :")
print(f"\nr² : {r2:.4f} ({r2*100:.1f}% de variance expliquee)")
print(f"\nerreur moyenne : {mean(erreurs):,.0f} euros")
print(f"erreur absolue moyenne : {mean(erreurs_absolues):,.0f} euros")
print(f"ecart-type des erreurs : {standard_deviation(erreurs):,.0f} euros")
print(f"\nerreur moyenne en % : {mean([abs(e) for e in erreurs_pct]):.1f}%")

# interpretation pour les agents
print("\ninterpretation pour les conseillers :")
print(f"en moyenne, le modele se trompe de {mean(erreurs_absolues):,.0f} euros")
print(f"soit environ {mean([abs(e) for e in erreurs_pct]):.1f}% du prix reel")
print(f"\ncela reste dans la marge de ±15% pour la majorite des estimations")

performance du modele sur les donnees d'entrainement :

r² : 0.5463 (54.6% de variance expliquee)

erreur moyenne : 0 euros
erreur absolue moyenne : 46,583 euros
ecart-type des erreurs : 64,204 euros

erreur moyenne en % : 10600.9%

interpretation pour les conseillers :
en moyenne, le modele se trompe de 46,583 euros
soit environ 10600.9% du prix reel

cela reste dans la marge de ±15% pour la majorite des estimations
